In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
train_data.head()

In [ ]:
test_data = pd.read_csv("/kaggle/input/titanic/test.csv")
const_test_data = pd.read_csv("/kaggle/input/titanic/test.csv")
test_data.head()

In [ ]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

In [ ]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

In [ ]:
plt.figure(figsize=(12, 2))
plt.scatter(train_data["Age"][train_data["Survived"] == 1],
         np.random.rand(train_data["Age"][train_data["Survived"] == 1].shape[0]), label='Survived')
plt.scatter(train_data["Age"][train_data["Survived"] == 0],
         np.random.rand(train_data["Age"][train_data["Survived"] == 0].shape[0]), label='Not Survived')
plt.legend();

In [ ]:
# fig = plt.figure(figsize=(12, 2))
# sns.displot(train_data["Age"][train_data["Survived"] == 1], kde=True);
# sns.displot(train_data["Age"][train_data["Survived"] == 0], kde=True);

In [ ]:
def model(X):
    return X["Age"] <= 40 

In [ ]:
bins = [0, 12, 18, 30, 50, 100]
labels = ['Child', 'Teen', 'Adult', 'Middle-aged', 'Senior']
train_data['AgeGroup'] = pd.cut(train_data['Age'], bins=bins, labels=labels)
test_data['AgeGroup'] = pd.cut(test_data['Age'], bins=bins, labels=labels)

train_data['FamilySize'] = train_data["SibSp"] + train_data["Parch"] + 1
test_data['FamilySize'] = test_data["SibSp"] + test_data["Parch"] + 1

In [ ]:
train_data

In [ ]:
from sklearn.preprocessing import LabelEncoder

for c in ["Sex", "Embarked", "AgeGroup"]:
    le = LabelEncoder().fit(train_data[c])
    train_data[c] = le.transform(train_data[c])
    test_data[c] = le.transform(test_data[c])

In [ ]:
train_data["Age"] = train_data["Age"].fillna(0)
test_data["Age"] = test_data["Age"].fillna(0)

test_data["Fare"] = test_data["Fare"].fillna(0)

In [ ]:
from sklearn.model_selection import train_test_split
y = train_data.Survived

features = ["Pclass", "Sex", "Age","SibSp", "Parch", "Fare", "Embarked", 
            "FamilySize", "AgeGroup"]
X = train_data[features]
X_test = test_data[features]

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score

model = RandomForestRegressor(random_state=1)
model.fit(train_X, train_y)
predictions = model.predict(val_X) > 0.5

print("Validation accuracy for Random Forest Model: ", accuracy_score(predictions, val_y))

In [ ]:
final_model = RandomForestRegressor()
final_model.fit(X, y)
final_predictions = (final_model.predict(X_test) > 0.5).astype(int)
print(final_predictions)

In [ ]:
output = pd.DataFrame({'PassengerId': const_test_data.PassengerId, 'Survived': final_predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")